In [1]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554977 sha256=395ab03a1de11afa5d08628660f782eafb63e93abbb7ac3cfeb4ff76cb1081ec
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

In [3]:
from google.colab import files
uploaded = files.upload()

Saving movie.csv to movie.csv


In [4]:
movie = pd.read_csv("movie.csv")

print(movie.head())

print(movie.shape)

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
(27278, 3)


In [5]:
# check missing values
print(movie.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64


In [6]:
# Remove duplicate rows

movie = movie.drop_duplicates()

# Fill missing values if any

movie.fillna('', inplace=True)

print(movie.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [7]:
movie['tags'] = movie['genres']

print(movie[['title', 'tags']].head())

                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   

                                          tags  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [8]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = cv.fit_transform(
    movie['tags']
).toarray()

print(vectors.shape)

(27278, 23)


In [9]:
#cosine similarity calculation
similarity = cosine_similarity(vectors)

print(similarity.shape)

(27278, 27278)


In [10]:
def recommend(movie_name):

    # Find movie index
    movie_index = movie[
        movie['title'] == movie_name
    ].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort movies based on similarity
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print("\nRecommended Movies:\n")

    # Print recommended movie titles
    for i in movie_list:
        print(
            movie.iloc[i[0]].title
        )

In [11]:
recommend("Toy Story (1995)")


Recommended Movies:

Antz (1998)
Toy Story 2 (1999)
Adventures of Rocky and Bullwinkle, The (2000)
Emperor's New Groove, The (2000)
Monsters, Inc. (2001)


In [12]:
!pip install scikit-surprise

In [13]:
!pip uninstall numpy -y

!pip install numpy==1.26.4

!pip install scikit-surprise

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 27.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have 

In [1]:
from surprise import Reader
from surprise import Dataset
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

In [1]:
from google.colab import files

uploaded = files.upload()

Saving u.data to u.data


In [2]:
import pandas as pd

# Load ratings dataset

ratings = pd.read_csv(
    "u.data",
    sep='\t',
    names=['userId', 'movieId', 'rating', 'timestamp']
)

print(ratings.head())

   userId  movieId  rating  timestamp
0       0       50       5  881250949
1       0      172       5  881250949
2       0      133       1  881250949
3     196      242       3  881250949
4     186      302       3  891717742


In [3]:
from google.colab import files

uploaded = files.upload()

Saving u.item to u.item


In [4]:
# Load movies dataset

movies = pd.read_csv(
    "u.item",
    sep='|',
    encoding='latin-1',
    header=None,
    usecols=[0, 1],
    names=['movieId', 'title']
)

print(movies.head())

   movieId              title
0        1   Toy Story (1995)
1        2   GoldenEye (1995)
2        3  Four Rooms (1995)
3        4  Get Shorty (1995)
4        5     Copycat (1995)


In [5]:
movie_data = pd.merge(
    ratings,
    movies,
    on='movieId'
)

print(movie_data.head())

   userId  movieId  rating  timestamp                            title
0       0       50       5  881250949                 Star Wars (1977)
1       0      172       5  881250949  Empire Strikes Back, The (1980)
2       0      133       1  881250949        Gone with the Wind (1939)
3     196      242       3  881250949                     Kolya (1996)
4     186      302       3  891717742         L.A. Confidential (1997)


In [6]:
movie_matrix = movie_data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

print(movie_matrix.head())

title   'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
userId                                                                   
0                             NaN           NaN                    NaN   
1                             NaN           NaN                    2.0   
2                             NaN           NaN                    NaN   
3                             NaN           NaN                    NaN   
4                             NaN           NaN                    NaN   

title   12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
userId                                                                 
0                       NaN         NaN                          NaN   
1                       5.0         NaN                          NaN   
2                       NaN         NaN                          NaN   
3                       NaN         2.0                          NaN   
4                       NaN         NaN          

In [7]:
toy_story_ratings = movie_matrix['Toy Story (1995)']

In [8]:
similar_movies = movie_matrix.corrwith(
    toy_story_ratings
)

corr_toy_story = pd.DataFrame(
    similar_movies,
    columns=['Correlation']
)

corr_toy_story.dropna(inplace=True)

print(
    corr_toy_story.sort_values(
        'Correlation',
        ascending=False
    ).head(10)
)

/usr/local/lib/python3.12/dist-packages/numpy/lib/function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


                                                    Correlation
title                                                          
Old Lady Who Walked in the Sea, The (Vieille qu...          1.0
Reckless (1995)                                             1.0
Ladybird Ladybird (1994)                                    1.0
Infinity (1996)                                             1.0
Albino Alligator (1996)                                     1.0
Toy Story (1995)                                            1.0
Guantanamera (1994)                                         1.0
Late Bloomers (1996)                                        1.0
Across the Sea of Time (1995)                               1.0
Substance of Fire, The (1996)                               1.0
